# List Buckets and Files in MinIO

This notebook shows you how to **connect to a MinIO server** and **list all buckets and files** stored in it.

## What is MinIO?

MinIO is an open-source object storage server that is compatible with Amazon S3.
Think of it as a file server in the cloud where files are organized into **buckets** (like folders).

## Prerequisites

- The `minio` Python package must be installed (`pip install minio`)
- You need the following **environment variables** set on your system:
  - `MINIO_ENDPOINT` — the server address (e.g. `minio.example.com:9000`)
  - `MINIO_ACCESS_KEY` — your access key (like a username)
  - `MINIO_SECRET_KEY` — your secret key (like a password)
  - `MINIO_SECURE` — optional, set to `true` to use HTTPS (defaults to `false`)

## Step 1 — Install the MinIO library

Run this cell to make sure the `minio` package is installed.

In [ ]:
# Install the MinIO Python client library
# The '!' prefix runs a shell command from inside a notebook
!pip install minio --quiet

## Step 2 — Import the libraries

We need two libraries:
- `os` — to read environment variables from the system
- `minio` — to talk to the MinIO server

In [ ]:
import os            # Built-in library to access environment variables
from minio import Minio  # MinIO client library

## Step 3 — Read connection settings from environment variables

Environment variables keep sensitive information (passwords, keys) **out of your code**.
They are set on the system and read at runtime, so you never have to write secrets in a notebook.

| Variable           | Description                                |
|--------------------|--------------------------------------------|  
| `MINIO_ENDPOINT`   | Server address, e.g. `minio.example.com:9000` |
| `MINIO_ACCESS_KEY` | Your access key (like a username)          |
| `MINIO_SECRET_KEY` | Your secret key (like a password)          |
| `MINIO_SECURE`     | `true` for HTTPS, `false` for HTTP (default) |

In [ ]:
# Read connection settings from environment variables.
# os.environ["VAR"] raises an error if the variable is missing — this is
# intentional so you notice right away if something is not configured.

endpoint   = os.environ["MINIO_ENDPOINT"]    # e.g. "minio.example.com:9000"
access_key = os.environ["MINIO_ACCESS_KEY"]   # your access key
secret_key = os.environ["MINIO_SECRET_KEY"]   # your secret key

# For the secure flag we use os.environ.get() which returns a default value
# instead of raising an error when the variable is not set.
secure     = os.environ.get("MINIO_SECURE", "false").lower() == "true"

print(f"MinIO endpoint : {endpoint}")
print(f"Secure (HTTPS) : {secure}")

## Step 4 — Connect to MinIO

Create a **client** object. This does not send a request yet — it just prepares the connection.

In [ ]:
# Create a MinIO client using the settings we read above.
# The client handles all communication with the MinIO server.

client = Minio(
    endpoint,               # server address
    access_key=access_key,   # authentication: who you are
    secret_key=secret_key,   # authentication: proof of identity
    secure=secure,           # True = HTTPS, False = HTTP
)

print("Client created successfully — ready to connect!")

## Step 5 — List all buckets

A **bucket** is a top-level container in object storage (similar to a folder).
Let's retrieve and display every bucket available on the server.

In [ ]:
# list_buckets() contacts the server and returns a list of all buckets
# that your access key is allowed to see.

buckets = client.list_buckets()

print(f"Found {len(buckets)} bucket(s):\n")

for bucket in buckets:
    # Each bucket has a name and a creation_date
    print(f"  - {bucket.name}  (created: {bucket.creation_date})")

## Step 6 — List files inside each bucket

Now we loop through every bucket and list the **objects** (files) it contains.

`list_objects()` returns an iterator — we go through each item and print its name and size.

In [ ]:
for bucket in buckets:
    print(f"\n{'='*60}")
    print(f"Bucket: {bucket.name}")
    print(f"{'='*60}")

    # list_objects() lists all objects in the bucket.
    # recursive=True means it also lists objects inside "sub-folders".
    objects = client.list_objects(bucket.name, recursive=True)

    count = 0
    for obj in objects:
        # obj.object_name = full path of the file
        # obj.size        = file size in bytes
        # obj.last_modified = when the file was last updated
        size_kb = (obj.size or 0) / 1024  # convert bytes to kilobytes
        print(f"  {obj.object_name:<50} {size_kb:>10.1f} KB   {obj.last_modified}")
        count += 1

    if count == 0:
        print("  (empty bucket — no files found)")
    else:
        print(f"  --- {count} file(s) total ---")

## Summary

In this notebook you learned how to:

1. **Read connection settings** from environment variables (keeping secrets out of code)
2. **Create a MinIO client** to communicate with the server
3. **List all buckets** on the server
4. **List all files** inside each bucket

### Next steps

- Upload a file: `client.fput_object("my-bucket", "remote-name.csv", "/local/path.csv")`
- Download a file: `client.fget_object("my-bucket", "remote-name.csv", "/local/path.csv")`
- Create a bucket: `client.make_bucket("new-bucket")`